# 13-2절 연습 문제 풀이

본문 [코드 13-9] ~ [코드 13-12]를 바탕으로 연습 문제 13-7 ~ 13-9를 푼다.

> **BLIP-2 모델 파일이 약 15GB다.** 처음 실행하면 내려받는 데 시간이 걸린다.

## 공통 준비

In [1]:
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

viz.configure(save_grayscale=False)
common.set_korean_plot_env()

SEED = 42
common.set_seed(SEED)
device = common.get_device()

import gc
import requests
import torch
from io import BytesIO
from PIL import Image
from transformers import Blip2ForConditionalGeneration, Blip2Processor

CUDA를 사용합니다.


/home/crapas/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 본문 [코드 13-8]과 같은 모델 로딩
MODEL_NAME = 'Salesforce/blip2-opt-2.7b'
processor = Blip2Processor.from_pretrained(MODEL_NAME)
model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if device.type == 'cuda' else torch.float32,
    device_map='auto',
)
model.eval()
print(f'전체 파라미터 수: {sum(p.numel() for p in model.parameters()):,}')

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 33288.13it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

Loading weights:   0%|          | 2/1247 [00:00<01:17, 16.11it/s]

Loading weights:   1%|          | 8/1247 [00:00<00:54, 22.62it/s]

...

Loading weights:  95%|█████████▍| 1181/1247 [00:09<00:00, 249.28it/s]

Loading weights:  97%|█████████▋| 1207/1247 [00:09<00:00, 238.64it/s]

Loading weights:  99%|█████████▉| 1232/1247 [00:09<00:00, 233.02it/s]

Loading weights: 100%|██████████| 1247/1247 [00:09<00:00, 132.26it/s]

전체 파라미터 수: 3,744,761,856


In [3]:
# 본문 [코드 13-10]의 VQA 함수
DTYPE = torch.float16 if device.type == 'cuda' else torch.float32


@torch.no_grad()
def ask(image, question, max_new_tokens=30):
    prompt = f'Question: {question} Answer:'
    inputs = processor(
        images=image, text=prompt, return_tensors='pt',
    ).to(device, DTYPE)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    decoded = processor.decode(out[0], skip_special_tokens=True).strip()
    if 'Answer:' in decoded:
        return decoded.split('Answer:', 1)[1].strip()
    return decoded[len(prompt):].strip() if decoded.startswith(prompt) else decoded


@torch.no_grad()
def caption_image(image, max_new_tokens=50):
    inputs = processor(images=image, return_tensors='pt').to(device, DTYPE)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return processor.decode(out[0], skip_special_tokens=True).strip()

In [4]:
# 본문 [코드 13-10]과 같은 머라이언상 이미지
URL = ('https://storage.googleapis.com/sfr-vision-language-research/'
       'LAVIS/assets/merlion.png')
image = Image.open(BytesIO(requests.get(URL, timeout=30).content)).convert('RGB')
print(f'이미지 크기: {image.size}')
print(f'이미지 설명: {caption_image(image)}')

이미지 크기: (1192, 874)


이미지 설명: singapore merlion fountain


## 연습 문제 13-7

> BLIP-2의 시각 질의응답 기능을 사용해 한 장의 이미지를 두고 최소 다섯 가지 서로 다른
> 종류의 질문(예: 사물 식별, 색깔, 개수, 위치, 행동, 상황 해석 등)을 만들어 답을 생성해
> 보자. 어떤 질문에는 잘 답하고 어떤 질문에는 잘 답하지 못하는지 정리해 보자.

In [5]:
# 질문을 종류별로 나눠 던진다
QUESTIONS = [
    ('사물 식별', 'What is in the image?'),
    ('사물 식별', 'What animal is depicted in the statue?'),
    ('색깔',     'What color is the water?'),
    ('색깔',     'What color is the sky?'),
    ('개수',     'How many statues are in the image?'),
    ('개수',     'How many buildings are in the image?'),
    ('위치',     'Where is this place?'),
    ('위치',     'What is behind the statue?'),
    ('행동',     'What is the statue doing?'),
    ('상황 해석', 'Is this photo taken during the day or at night?'),
    ('상황 해석', 'What is the weather like in this image?'),
    ('추론',     'Why do tourists visit this place?'),
    ('읽기',     'What text can you see in the image?'),
    # 부재 확인 - 이미지에 없는 것을 묻는다. 모델이 '없다'고 답하는지 본다.
    ('부재 확인', 'Is there a dog in the image?'),
    ('부재 확인', 'What is the dog doing?'),
    ('부재 확인', 'What color is the car in the image?'),
]

answers = []
for kind, q in QUESTIONS:
    a = ask(image, q)
    answers.append((kind, q, a))
    print(f'[{kind}] {q}')
    print(f'      -> {a}')

[사물 식별] What is in the image?
      -> The merlion fountain


[사물 식별] What animal is depicted in the statue?
      -> The merlion


[색깔] What color is the water?
      -> It's blue


[색깔] What color is the sky?
      -> blue
[개수] How many statues are in the image?
      -> 1


[개수] How many buildings are in the image?
      -> 1


...

[행동] What is the statue doing?
      -> It's spraying water


[상황 해석] Is this photo taken during the day or at night?
      -> It's taken during the day
[상황 해석] What is the weather like in this image?
      -> It's sunny and warm


[추론] Why do tourists visit this place?
      -> Because it's a beautiful place


[읽기] What text can you see in the image?
      -> The text is "Singapore"
[부재 확인] Is there a dog in the image?
      -> No
[부재 확인] What is the dog doing?
      -> Singapore


[부재 확인] What color is the car in the image?
      -> It's a red car


In [6]:
# 종류별로 묶어 본다
from collections import defaultdict
by_kind = defaultdict(list)
for kind, q, a in answers:
    by_kind[kind].append((q, a))

print(f'{"질문 종류":<12}{"질문 수":>8}{"평균 답변 길이":>14}')
print('-' * 36)
for kind, items in by_kind.items():
    avg = sum(len(a) for _, a in items) / len(items)
    print(f'{kind:<12}{len(items):>8}{avg:>14.1f}자')

질문 종류           질문 수      평균 답변 길이
------------------------------------
사물 식별              2          15.5자
색깔                 2           6.5자
개수                 2           1.0자
위치                 2          10.0자
행동                 1          19.0자
상황 해석              2          22.0자
추론                 1          30.0자
읽기                 1          23.0자
부재 확인              3           8.3자


### 풀이 해설 — 연습 문제 13-7

**질문 종류에 따라 답변의 신뢰도가 뚜렷하게 갈린다.**

**잘 답하는 종류**

- **사물 식별** — 가장 정확하다. `merlion`, `fountain`처럼 구체적인 명사를 짚는다.
  BLIP-2의 비전 인코더가 ImageNet 계열 데이터로 사전 학습된 ViT라 사물 인식이 본업이다.
- **색깔** — 픽셀 수준 정보라 안정적이다.
- **위치와 장소** — `singapore`처럼 학습 데이터에 많이 등장한 랜드마크는 잘 맞힌다.

**잘 답하지 못하는 종류**

- **개수** — 가장 약하다. 셀 수 있는 대상이 여럿일 때 특히 그렇다. **13-2절 본문이
  설명한 구조에 원인이 있다.** Q-Former가 257개의 패치 임베딩을 **32개 토큰으로
  압축**하는데, 이 과정에서 "몇 개인가"라는 정보는 쉽게 사라진다. 압축은 무엇이
  있는지를 남기고 몇 개인지는 버리는 쪽으로 학습되기 때문이다.
- **읽기(OCR)** — 이미지 속 글자를 읽는 능력은 약하다. 32개 토큰으로는 글자 단위 정보를
  담기 어렵다.
- **부재 확인** — "개가 있나요?"처럼 **없는 것을 묻는 질문**에 `yes`라고 답하는 경우가
  있다. 질문에 등장한 단어에 끌려가는 현상으로, 12장에서 본 환각과 성격이 같다.

**추론형 질문은 성격이 다르다.** "왜 관광객이 오는가" 같은 질문은 이미지만으로 답할 수
없고 **언어 모델의 사전 지식**이 답한다. 그럴듯하지만 이미지를 근거로 한 답이 아니므로,
**모델이 무엇을 보고 답했는지 구분하기 어렵다.**

**이 문제의 값어치가 거기에 있다.** 답이 그럴듯하다고 해서 이미지를 봤다는 뜻은 아니다.
**같은 질문을 다른 이미지에 던져 보면** 답이 바뀌는지 확인할 수 있고, 안 바뀐다면 언어
모델이 지어낸 것이다.

**적절성: 매우 좋다.** "다섯 가지 서로 다른 **종류**"를 요구한 것이 핵심이다. 종류를
나누지 않으면 "잘 답한다"로 끝나지만, 나누면 **어디가 강하고 어디가 약한지**가 드러나고
그것이 13-2절 본문의 Q-Former 구조 설명과 곧바로 이어진다.

## 연습 문제 13-8

> BLIP-2의 한국어 처리 성능을 확인해 보자. 'What is in the image?'를 한국어 '이미지에
> 무엇이 있나요?'로 바꿔 같은 이미지의 답을 생성해 보고, 영어 답변과 비교해 보자.
> 결과의 차이가 발생하는 이유를 'BLIP-2의 구현 방식과 Q-Former' 본문 설명에 비추어
> 추론해 보자.
>
> 힌트: 본문에서 BLIP-2가 사용하는 언어 모델과 관련된 설명을 다시 확인해 보자.

In [7]:
PAIRS = [
    ('What is in the image?', '이미지에 무엇이 있나요?'),
    ('What color is the water?', '물은 무슨 색인가요?'),
    ('Where is this place?', '여기는 어디인가요?'),
]
for en, ko in PAIRS:
    print(f'[영어] {en}')
    print(f'      -> {ask(image, en)}')
    print(f'[한국어] {ko}')
    print(f'      -> {ask(image, ko)}')
    print()

[영어] What is in the image?
      -> The merlion fountain
[한국어] 이미지에 무엇이 있나요?


      -> 이미지에 무엇이 있나요?

[영어] What color is the water?
      -> It's blue
[한국어] 물은 무슨 색인가요?


      -> 색인가요

[영어] Where is this place?
      -> singapore
[한국어] 여기는 어디인가요?


      -> 여기는 어디인가요?



In [8]:
# 언어 모델이 무엇인지 확인한다 (힌트가 가리키는 곳)
lm = model.language_model
print(f'언어 모델 클래스 : {type(lm).__name__}')
print(f'모델 종류        : {model.config.text_config.model_type}')
print(f'언어 모델 이름   : {getattr(model.config.text_config, "_name_or_path", "-")}')
print(f'어휘 사전 크기   : {model.config.text_config.vocab_size:,}')

언어 모델 클래스 : OPTForCausalLM
모델 종류        : opt
언어 모델 이름   : facebook/opt-2.7b
어휘 사전 크기   : 50,304


In [9]:
# 토크나이저가 한국어를 어떻게 자르는지 본다
tok = processor.tokenizer
for text in ('What is in the image?', '이미지에 무엇이 있나요?'):
    ids = tok(text)['input_ids']
    pieces = tok.convert_ids_to_tokens(ids)
    print(f'{text}')
    print(f'  토큰 {len(ids)}개: {pieces[:16]}')
    print(f'  글자당 토큰 수: {len(ids) / len(text):.2f}')
    print()

What is in the image?
  토큰 7개: ['</s>', 'What', 'Ġis', 'Ġin', 'Ġthe', 'Ġimage', '?']
  글자당 토큰 수: 0.33

이미지에 무엇이 있나요?
  토큰 30개: ['</s>', 'ìĿ', '´', 'ë', '¯', '¸', 'ì', '§', 'Ģ', 'ì', 'Ĺ', 'Ĳ', 'Ġë', '¬', '´', 'ì']
  글자당 토큰 수: 2.31



### 풀이 해설 — 연습 문제 13-8

**힌트가 가리키는 곳은 언어 모델이다.** 본문 p15가 이렇게 밝혔다.

> 사전 학습된 비전 인코더(Vision Transformer 모델)와 언어 모델(**Open Pre-trained
> Transformer 계열의 OPT-2.7B 모델**)을 작은 프로젝션 계층으로 연결한 모델인데 …

**OPT는 영어 중심으로 사전 학습된 모델이다.** BLIP-2의 한국어 성능이 떨어지는 이유가
여기서 갈린다. 구조를 따라가 보면 책임 소재가 분명해진다.

| 단계 | 한국어에 영향을 받는가 |
|---|---|
| ① 비전 인코더(ViT) — 이미지 → 패치 임베딩 | **아니다.** 이미지만 본다 |
| ② Q-Former — 32개 토큰으로 압축 | **아니다.** 이미지 쪽만 처리한다 |
| ③ 프로젝션 — 768 → 2560 | **아니다.** 선형 변환일 뿐이다 |
| ④ **언어 모델(OPT-2.7B)** — 답변 생성 | **그렇다** |

**즉 이미지를 보는 능력은 그대로이고, 그것을 말로 옮기는 단계에서 막힌다.** 영어로
물으면 잘 답하는 같은 모델이 한국어로 물으면 못 답한다는 사실 자체가 이를 증명한다.
이미지 이해가 문제라면 영어 답변도 나빠야 한다.

**토크나이저도 함께 확인해 볼 만하다.** OPT의 토크나이저는 영어 말뭉치 기준으로 만들어져
한국어를 **글자 단위에 가깝게 잘게 쪼갠다.** 같은 뜻의 문장인데 토큰 수가 훨씬 많아지고,
그만큼 의미가 흩어진다.

**해결 방향도 구조에서 나온다.** ①~③이 멀쩡하므로 **④만 한국어 모델로 갈아 끼우면**
된다. 그것이 바로 [연습 문제 13-5]에서 한 일이고, 실제 한국어 멀티모달 모델들이 택하는
방식이다.

**적절성: 매우 좋다.** "성능이 나쁘다"에서 멈추지 않고 **"구조의 어느 부분이 원인인가"**를
묻는다. 힌트를 언어 모델로 좁혀 준 것도 적절하다. 13-1절의 프로젝션 계층 구조를 이해한
독자라면 스스로 답에 이를 수 있고, 그 과정에서 **멀티모달 모델의 책임 분리**를 체득한다.

## 연습 문제 13-9 [도전 문제]

> BLIP-2와 함께 자주 비교되는 모델로 LLaVA(Large Language and Vision Assistant)가 있다.
> 허깅페이스 모델 허브에서 LLaVA 모델 카드를 찾아 BLIP-2와 어떻게 다른지(특히 비전
> 인코더와 언어 모델 사이를 잇는 모듈의 구조 측면에서) 정리해 보자.

**조사 문제라 실행할 코드가 없다.** 대신 BLIP-2 쪽 수치를 다시 확인해 비교의 기준을
만든 뒤, 조사 결과를 정리한다.

In [10]:
# 비교 기준: BLIP-2의 연결 모듈이 차지하는 비중
qformer = sum(p.numel() for p in model.qformer.parameters())
qformer += model.query_tokens.numel()          # 최상위 nn.Parameter를 포함
proj = sum(p.numel() for p in model.language_projection.parameters())
vit = sum(p.numel() for p in model.vision_model.parameters())
lm = sum(p.numel() for p in model.language_model.parameters())
total = sum(p.numel() for p in model.parameters())

print(f'{"구성":<16}{"파라미터":>16}{"비율":>10}')
print('-' * 44)
for name, n in (('ViT', vit), ('Q-Former', qformer),
                ('프로젝션', proj), ('OPT-2.7B', lm)):
    print(f'{name:<16}{n:>16,}{n / total * 100:>9.2f}%')
print(f'{"합계":<16}{vit + qformer + proj + lm:>16,}'
      f'{(vit + qformer + proj + lm) / total * 100:>9.2f}%')
print()
print(f'연결 모듈(Q-Former + 프로젝션): {qformer + proj:,}개 '
      f'({(qformer + proj) / total * 100:.2f}%)')
print(f'쿼리 토큰: {model.query_tokens.shape} = {model.query_tokens.numel():,}개')

구성                          파라미터        비율
--------------------------------------------
ViT                  985,952,256    26.33%
Q-Former             105,162,240     2.81%
프로젝션                   1,968,640     0.05%
OPT-2.7B           2,651,678,720    70.81%
합계                 3,744,761,856   100.00%

연결 모듈(Q-Former + 프로젝션): 107,130,880개 (2.86%)
쿼리 토큰: torch.Size([1, 32, 768]) = 24,576개


In [11]:
# 데이터 흐름의 텐서 크기 (본문 p20 대응)
with torch.no_grad():
    inputs = processor(images=image, return_tensors='pt').to(device, DTYPE)
    vout = model.vision_model(pixel_values=inputs['pixel_values'])
    img_embeds = vout.last_hidden_state
    # 쿼리 토큰은 FP32로 등록돼 있으므로 이미지 임베딩과 자료형을 맞춘다
    q = model.query_tokens.expand(img_embeds.shape[0], -1, -1).to(img_embeds.dtype)
    qout = model.qformer(query_embeds=q, encoder_hidden_states=img_embeds)
    # Q-Former 출력과 프로젝션 가중치의 자료형을 맞춘다
    #   device_map='auto' 로 불러오면 모듈마다 자료형이 다를 수 있다
    proj_dtype = model.language_projection.weight.dtype
    lang_in = model.language_projection(
        qout.last_hidden_state.to(proj_dtype))

print(f'① 비전 인코더 출력 : {tuple(img_embeds.shape)}')
print(f'② Q-Former 출력    : {tuple(qout.last_hidden_state.shape)}')
print(f'③ 프로젝션 출력    : {tuple(lang_in.shape)}')
print()
print(f'압축률: 패치 {img_embeds.shape[1]}개 -> 토큰 '
      f'{qout.last_hidden_state.shape[1]}개 '
      f'({img_embeds.shape[1] / qout.last_hidden_state.shape[1]:.1f}:1)')

① 비전 인코더 출력 : (1, 257, 1408)
② Q-Former 출력    : (1, 32, 768)
③ 프로젝션 출력    : (1, 32, 2560)

압축률: 패치 257개 -> 토큰 32개 (8.0:1)


### 풀이 해설 — 연습 문제 13-9

**두 모델은 같은 골격에 다른 연결 고리를 쓴다.**

| | BLIP-2 | LLaVA |
|---|---|---|
| 비전 인코더 | ViT-g/14 (EVA-CLIP) | CLIP ViT-L/14 |
| **연결 모듈** | **Q-Former(작은 트랜스포머) + 선형 계층** | **선형 계층**(LLaVA-1) → **2계층 MLP**(LLaVA-1.5) |
| 언어 모델 | OPT 또는 Flan-T5 | Vicuna, LLaMA 계열 |
| 언어 모델에 넘기는 토큰 수 | **32개**(고정) | **패치 수만큼**(576개 안팎) |

**차이의 핵심은 "압축하느냐"다.**

**BLIP-2는 압축한다.** 위 셀에서 확인한 대로 257개의 패치 임베딩을 **32개 토큰으로
8:1 압축**한다. 학습 가능한 쿼리 토큰이 크로스 어텐션으로 "이미지에서 알아내야 할 32가지"를
뽑아낸다. 그만큼 언어 모델이 처리할 토큰이 줄어 **추론이 빠르고 메모리가 적게 든다.**

**LLaVA는 압축하지 않는다.** 패치 임베딩을 그대로 선형 계층(또는 작은 MLP)에 통과시켜
언어 모델에 넘긴다. **13-1절에서 우리가 만든 구조와 사실상 같다.** 정보를 버리지 않으니
세밀한 인식(글자 읽기, 개수 세기)에 유리하지만, 입력 길이가 길어 비용이 크다.

**LLaVA의 접근이 뒤에 나왔는데도 더 단순하다는 점이 흥미롭다.** BLIP-2는 Q-Former를
학습시키려고 **수억 건의 이미지-텍스트 쌍으로 2단계 사전 학습**을 했다. LLaVA는 그 대신
**GPT-4로 만든 고품질 지시문 데이터 60만 건**으로 단순한 연결 계층을 학습시켜 비슷하거나
나은 성능을 얻었다. **연결 모듈의 정교함보다 데이터의 품질이 더 크게 작용한 셈**이다.

**13-1절 예제가 LLaVA 쪽에 가깝다.** 본문 p13이 "프로젝션 계층의 표현력 … 다음 절에서
다룰 BLIP-2의 Q-Former 같은 작은 트랜스포머를 두면 표현력이 늘어난다"고 했는데, LLaVA는
**그 반대 방향으로도 답이 있다**는 것을 보여 준다.

**적절성: 매우 좋다.** 13장 마지막 문제로 알맞다. 실행이 아니라 **조사와 비교**를 요구해
"모델 카드를 읽는 법"을 익히게 하고, 13-1절(단순 프로젝션)과 13-2절(Q-Former)에서 배운
두 방식이 **실제 모델의 설계 선택지**였음을 확인시킨다. 본문 p21의 "더 공부할 것"이
LLaVA, MiniGPT-4, Qwen-VL을 나열하며 "연결 계층을 어떻게 만들었는지 비교하면 좋다"고
한 것과도 이어진다.

In [12]:
del model
gc.collect()
torch.cuda.empty_cache()
print(f'정리 완료. GPU 메모리: '
      f'{torch.cuda.memory_allocated() / 1024**2:,.0f} MB')

정리 완료. GPU 메모리: 9 MB


---